# FER2013 – CNN (PyTorch)

## Cách chạy

1. Cài dependency nếu chưa có: `pip install -r code/requirements.txt`.
2. Chạy toàn bộ cell.
3. Kết quả ở `Code_FER2013/runs/`, `figures/`, `experiment_log.csv`,
   `best_report.json`; file zip ở cuối notebook.

## Code và dữ liệu

- Notebook phải được chạy từ thư mục `Code_FER2013/`.
- `CODE = ./code`, `EXP_DIR = .`, `DATA_DIR = ../` (tức `BT02/`).
- Nếu kernel có cwd khác, sửa `CODE` trong cell cấu hình thành đường dẫn tuyệt đối
  tới `.../Code_FER2013/code`; `EXP_DIR` và `DATA_DIR` tự suy ra đường dẫn.
- Dữ liệu: dùng `BT02/fer2013.npz` nếu đã có. Nếu chưa, notebook sẽ tự động tải xuống.
  `icml_face_data.csv` vào `BT02/` rồi tạo npz. Có thể đặt sẵn CSV để chạy offline.
- Tùy chọn: `EPOCHS_MODE = "full"` (12/12/12/12/12/10 epoch) hoặc `"quick"`
  (`QUICK_EPOCHS` mỗi cấu hình), `FORCE_RERUN`, `RUN_ANALYSIS`.


## 1. Kiểm tra môi trường

In [ ]:
import sys, torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No CUDA: training will use CPU.")


## 2. Cấu hình

In [ ]:
import os
import sys

# Chạy notebook từ thư mục Code_FER2013/.
CODE = os.path.abspath("code")
EXP_DIR = os.path.dirname(CODE)
DATA_DIR = os.path.dirname(EXP_DIR)

sys.path.insert(0, CODE)  

DATA_CSV = os.path.join(DATA_DIR, "icml_face_data.csv")
DATA_NPZ = os.path.join(DATA_DIR, "fer2013.npz")

EPOCHS_MODE = "full"      # "full" = 12/12/12/12/12/10 epochs, "quick" = QUICK_EPOCHS each
QUICK_EPOCHS = 2
FORCE_RERUN = False       # True: retrain even if runs/<name>.json already exists
RUN_ANALYSIS = True

print("CODE:", CODE)
print("EXP_DIR:", EXP_DIR)
print("DATA_NPZ:", DATA_NPZ)


## 3. Kiểm tra code local

In [ ]:
import os

print("Using local code:", CODE)
print("Files:", sorted(os.listdir(CODE)))
assert os.path.isfile(os.path.join(CODE, "fer", "config.py")), f"Không thấy code tại {CODE}"
assert os.path.isfile(os.path.join(CODE, "train.py"))
assert os.path.isfile(os.path.join(CODE, "analyze.py"))


## 4. Dữ liệu FER2013

In [ ]:
import json, os, urllib.request

OID = "2c7ce224ef03ec2e2e522d10b93a069900b26fa1c028174d91d5f1f8013a2884"
SIZE = 301072768
REPO = "kelacloiloilac/Thuchanhvabaitapdeeplearning"


def download_lfs(oid, size, dest):
    payload = {"operation": "download", "transfers": ["basic"],
               "objects": [{"oid": oid, "size": size}]}
    url = f"https://github.com/{REPO}.git/info/lfs/objects/batch"
    headers = {"Accept": "application/vnd.git-lfs+json",
               "Content-Type": "application/vnd.git-lfs+json"}
    req = urllib.request.Request(url, data=json.dumps(payload).encode("utf-8"),
                                 headers=headers, method="POST")
    with urllib.request.urlopen(req, timeout=120) as r:
        info = json.load(r)
    href = info["objects"][0]["actions"]["download"]["href"]
    total = 0
    with urllib.request.urlopen(href, timeout=300) as resp, open(dest, "wb") as f:
        while True:
            chunk = resp.read(1 << 20)
            if not chunk:
                break
            f.write(chunk)
            total += len(chunk)
    return total


if os.path.exists(DATA_NPZ) and os.path.getsize(DATA_NPZ) > 1_000_000:
    print("Existing npz found; skip CSV download:", DATA_NPZ)
else:
    os.makedirs(DATA_DIR, exist_ok=True)
    if os.path.exists(DATA_CSV) and os.path.getsize(DATA_CSV) == SIZE:
        print("Dataset already present:", DATA_CSV)
    else:
        try:
            got = download_lfs(OID, SIZE, DATA_CSV)
            print("Downloaded", got, "bytes to", DATA_CSV)
        except Exception as e:
            raise RuntimeError(
                "Không tải được dữ liệu. Hãy đặt file icml_face_data.csv (FER2013) "
                f"vào {DATA_CSV} rồi chạy lại cell này."
            ) from e

    print("CSV size:", os.path.getsize(DATA_CSV))
    assert os.path.getsize(DATA_CSV) == SIZE, "Unexpected CSV size"


## 5. Tạo `fer2013.npz`

In [ ]:
import os, subprocess, sys

if os.path.exists(DATA_NPZ) and os.path.getsize(DATA_NPZ) > 1_000_000:
    print("Using existing", DATA_NPZ)
else:
    subprocess.run([sys.executable, "prepare_data.py", "--csv", DATA_CSV, "--out", DATA_NPZ],
                   cwd=CODE, check=True)

print("NPZ:", DATA_NPZ, os.path.getsize(DATA_NPZ), "bytes")


## 6. Huấn luyện E1–E6

In [ ]:
import json, os, subprocess, sys, time, torch

from fer.experiments import EXPERIMENTS

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", DEVICE)
if DEVICE == "cpu" and EPOCHS_MODE == "full":
    print("No CUDA: full mode is slow on CPU; set EPOCHS_MODE = 'quick' or use a GPU machine.")

runs_dir = os.path.join(EXP_DIR, "runs")
os.makedirs(runs_dir, exist_ok=True)
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
t_start = time.time()

for cfg in EXPERIMENTS:
    name = cfg["name"]
    out_json = os.path.join(runs_dir, name + ".json")
    pt_path = os.path.join(runs_dir, name + ".pt")
    default_epochs = cfg.get("epochs", 15)
    epochs = default_epochs if EPOCHS_MODE == "full" else min(default_epochs, QUICK_EPOCHS)
    want = dict(arch=cfg.get("arch", "small"), aug=cfg.get("aug", 1), epochs=epochs,
                lr=cfg.get("lr", 1e-3), opt=cfg.get("opt", "adam"),
                sched=cfg.get("sched", "none"), bs=cfg.get("bs", 64),
                wd=cfg.get("wd", 0.0), ls=cfg.get("ls", 0.0), device=DEVICE)
    if os.path.exists(out_json) and os.path.exists(pt_path) and not FORCE_RERUN:
        try:
            prev = json.load(open(out_json)).get("args", {})
        except Exception:
            prev = {}
        if all(prev.get(k) == v for k, v in want.items()):
            print("[skip]", name, "- matches", want)
            continue
        print("[retrain]", name, "| existing", {k: prev.get(k) for k in want},
              "-> requested", want)
    cmd = [sys.executable, "train.py", "--name", name,
           "--arch", cfg.get("arch", "small"), "--aug", str(cfg.get("aug", 1)),
           "--epochs", str(epochs), "--device", DEVICE,
           "--data", DATA_NPZ, "--out", runs_dir]
    for key in ("lr", "opt", "sched", "bs", "wd", "ls"):
        if key in cfg:
            cmd += [f"--{key}", str(cfg[key])]
    print("=" * 78)
    print("RUN", name, "| epochs", epochs, "| device", DEVICE)
    print("=" * 78, flush=True)
    t0 = time.time()
    subprocess.run(cmd, cwd=CODE, check=True, env=env)
    print(f"[done] {name} in {(time.time() - t0) / 60:.1f} min", flush=True)

print(f"All requested experiments finished in {(time.time() - t_start) / 60:.1f} min")


## 7. Phân tích, confusion matrix, filter

In [ ]:
import subprocess, sys

if RUN_ANALYSIS:
    subprocess.run([sys.executable, "analyze.py", "--data", DATA_NPZ], cwd=CODE, check=True)
    print("Analysis done:", EXP_DIR)
else:
    print("RUN_ANALYSIS is False; skipped")


## 8. Hiển thị kết quả

In [ ]:
import csv, json, os
from IPython.display import Image, Markdown, display

log_path = os.path.join(EXP_DIR, "experiment_log.csv")
report_path = os.path.join(EXP_DIR, "best_report.json")
fig_dir = os.path.join(EXP_DIR, "figures")


def show_results():
    display(Markdown("## Experiment log"))
    with open(log_path, encoding="utf-8-sig") as f:
        rows = list(csv.reader(f))
    if rows:
        header, body = rows[0], rows[1:]
        lines = ["| " + " | ".join(header) + " |",
                 "| " + " | ".join(["---"] * len(header)) + " |"]
        lines += ["| " + " | ".join(r) + " |" for r in body]
        display(Markdown("\n".join(lines)))

    report = json.load(open(report_path))
    display(Markdown(f"## Best: `{report['best']}` — PrivateTest accuracy **{report['test_acc'] * 100:.2f}%**"))
    print(f"accuracy = {report['report'].get('accuracy', 0):.4f}")
    for cls, m in report["report"].items():
        if isinstance(m, dict):
            print(f"{cls:12s} precision={m['precision']:.3f} recall={m['recall']:.3f} "
                  f"f1={m['f1-score']:.3f} support={int(m['support'])}")

    for fname in ["compare_val.png", "confusion_matrix.png", "filters_first_conv.png",
                  "filters_last_conv_weights.png", "filters_last_conv_actmax.png",
                  "feature_maps_last_conv.png", "wrong_cases.png", "true_cases.png"]:
        path = os.path.join(fig_dir, fname)
        if os.path.exists(path):
            display(Markdown(f"### {fname}"))
            display(Image(filename=path))


if os.path.exists(log_path) and os.path.exists(report_path):
    show_results()
else:
    print("No analysis outputs found. Set RUN_ANALYSIS=True and rerun the analysis cell.")


## 9. Đóng gói kết quả

In [ ]:
import os, zipfile
from IPython.display import FileLink, display

zip_path = os.path.join(EXP_DIR, "fer2013_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["runs", "figures", "experiment_log.csv", "best_report.json"]:
        path = os.path.join(EXP_DIR, name)
        if os.path.isdir(path):
            for root, _, files in os.walk(path):
                for f in files:
                    fp = os.path.join(root, f)
                    z.write(fp, os.path.relpath(fp, EXP_DIR))
        elif os.path.exists(path):
            z.write(path, name)

print("Archive:", zip_path, os.path.getsize(zip_path), "bytes")
try:
    display(FileLink(zip_path))
except Exception as e:
    print("Open manually:", zip_path, "|", e)
